# Task 4: General Health Query Chatbot (Prompt Engineering Based)
**Internship:** DevelopersHub Corporation — AI/ML Engineering  
**Objective:** Create a chatbot that answers general health-related questions using an LLM, with prompt engineering and safety filters.

**Tools Used:**
- Groq API (free) with `llama-3.3-70b-versatile` model
- Prompt Engineering for friendly medical assistant persona
- Safety filters to block harmful queries

## Step 1: Install Required Library

In [2]:
# Install groq library
!pip install groq -q

## Step 2: Import Libraries & Set API Key

In [ ]:
from groq import Groq

# -----------------------------------------------
# Paste your FREE Groq API key below
# Get it from: https://console.groq.com
# -----------------------------------------------
GROQ_API_KEY =" your  api  key  here "

# Initialize Groq client
client = Groq(api_key=GROQ_API_KEY)

print("✅ Groq client initialized successfully!")

## Step 3: Prompt Engineering
We design a **system prompt** that makes the LLM act like a friendly, professional medical assistant.

This is the core of **Prompt Engineering** — giving the model a clear persona, rules, and output format.

In [3]:
# ── SYSTEM PROMPT (Prompt Engineering) ──
# This defines the chatbot's persona, behavior, and response rules

SYSTEM_PROMPT = """
You are MediBot, a friendly and knowledgeable general health information assistant.

Your persona:
- Warm, empathetic, and easy to understand — like a helpful doctor friend
- Use simple, clear language that anyone can understand
- Structure responses in short paragraphs or bullet points
- Keep answers concise (around 150-200 words)

Your strict rules:
1. Provide GENERAL health information only — never diagnose or prescribe for a specific person
2. Always recommend consulting a real doctor for personal medical concerns
3. If asked about emergency symptoms (chest pain, stroke, severe bleeding), say: call emergency services immediately
4. Do NOT recommend specific drug dosages — refer to a pharmacist or doctor
5. Be honest if you are unsure about something
6. Maintain a positive, supportive, and professional tone at all times

Format: Use **bold** for key medical terms. End with a short friendly reminder to consult a healthcare professional.
"""

print("✅ System prompt defined!")
print(f"Prompt length: {len(SYSTEM_PROMPT)} characters")

✅ System prompt defined!
Prompt length: 954 characters


## Step 4: Safety Filters
Before sending any query to the LLM, we check it against safety rules to block harmful requests.

In [4]:
import re

# ── SAFETY FILTER PATTERNS ──
# These patterns detect dangerous or harmful queries

DANGER_PATTERNS = [
    r'overdose',
    r'how (much|many).*(kill|die|suicide)',
    r'want to (die|kill|hurt)',
    r'self.?harm',
    r'suicide',
    r'end my life',
    r'how to (make|create|synthesize).*(drug|poison|meth)',
    r'illegal drug',
]

# These patterns detect queries needing professional advice (not dangerous, but redirect needed)
ADVISORY_PATTERNS = [
    r'diagnose me',
    r'do i have',
    r'am i sick',
    r'what medicine should i take',
    r'prescribe',
    r'exact dosage for',
]

def check_safety(user_input):
    """
    Checks user input against safety patterns.
    Returns: 'danger', 'advisory', or 'safe'
    """
    text = user_input.lower()
    
    for pattern in DANGER_PATTERNS:
        if re.search(pattern, text):
            return 'danger'
    
    for pattern in ADVISORY_PATTERNS:
        if re.search(pattern, text):
            return 'advisory'
    
    return 'safe'


# Safety responses
DANGER_RESPONSE = """
⚠️  SAFETY FILTER ACTIVATED

I'm not able to help with that request as it may involve self-harm or dangerous activities.

If you or someone you know is in crisis, please reach out immediately:
• Emergency Services: 115 (Pakistan) / 911
• Visit your nearest hospital emergency department
• Talk to a trusted person or mental health professional

Your safety matters. Please seek help. 💙
"""

ADVISORY_RESPONSE = """
ℹ️  I understand you're looking for specific guidance.

As a general health information assistant, I cannot provide personal diagnoses 
or specific prescriptions — that requires a qualified doctor who can examine you properly.

I can share general information about conditions, symptoms, or healthy habits.
Please consult a healthcare professional for your specific situation. 🩺
"""

print("✅ Safety filters loaded!")
print(f"Danger patterns: {len(DANGER_PATTERNS)}")
print(f"Advisory patterns: {len(ADVISORY_PATTERNS)}")

# Test the safety filter
test_cases = [
    ("What causes a sore throat?", "Should be: safe"),
    ("I want to overdose",         "Should be: danger"),
    ("Diagnose me please",         "Should be: advisory"),
]

print("\n── Safety Filter Test ──")
for query, expected in test_cases:
    result = check_safety(query)
    print(f'  "{query}" → {result.upper()}  ({expected})')

✅ Safety filters loaded!
Danger patterns: 8
Advisory patterns: 6

── Safety Filter Test ──
  "What causes a sore throat?" → SAFE  (Should be: safe)
  "I want to overdose" → DANGER  (Should be: danger)
  "Diagnose me please" → ADVISORY  (Should be: advisory)


## Step 5: Core Chatbot Function
This function takes a user query, runs it through safety filters, then sends it to the LLM.

In [5]:
def ask_medibot(user_query, conversation_history=None):
    """
    Sends a health query to MediBot and returns the response.
    
    Args:
        user_query (str): The user's health question
        conversation_history (list): Previous messages for context (optional)
    
    Returns:
        str: MediBot's response
    """
    print(f"\n{'='*55}")
    print(f"🧑 User: {user_query}")
    print(f"{'='*55}")
    
    # Step 1: Run safety check
    safety_level = check_safety(user_query)
    
    if safety_level == 'danger':
        print("🛡️  [Safety Filter] DANGER query blocked.")
        print(DANGER_RESPONSE)
        return DANGER_RESPONSE
    
    if safety_level == 'advisory':
        print("ℹ️  [Safety Filter] Advisory redirect applied.")
        print(ADVISORY_RESPONSE)
        return ADVISORY_RESPONSE
    
    # Step 2: Build messages list
    messages = []
    
    # Add conversation history if provided
    if conversation_history:
        messages.extend(conversation_history)
    
    # Add current user query
    messages.append({"role": "user", "content": user_query})
    
    # Step 3: Call Groq API
    print("🤖 MediBot: ", end="")
    
    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT}
        ] + messages,
        max_tokens=600,
        temperature=0.65,
    )
    
    reply = response.choices[0].message.content
    print(reply)
    
    return reply

print("✅ ask_medibot() function ready!")

✅ ask_medibot() function ready!


## Step 6: Test with Example Queries
Testing all example queries mentioned in the task requirements.

In [ ]:
# ── EXAMPLE QUERY 1 (from PDF requirements) ──
response1 = ask_medibot("What causes a sore throat?")

In [ ]:
# ── EXAMPLE QUERY 2 (from PDF requirements) ──
response2 = ask_medibot("Is paracetamol safe for children?")

In [ ]:
# ── EXTRA QUERY 3 ──
response3 = ask_medibot("How can I lower my blood pressure naturally?")

In [ ]:
# ── EXTRA QUERY 4 ──
response4 = ask_medibot("What are the symptoms of diabetes?")

## Step 7: Safety Filter Tests
Demonstrating that harmful queries are blocked.

In [8]:
# ── DANGEROUS QUERY — should be blocked ──
response_blocked = ask_medibot("How do I overdose on medicine?")


🧑 User: How do I overdose on medicine?
🛡️  [Safety Filter] DANGER query blocked.

⚠️  SAFETY FILTER ACTIVATED

I'm not able to help with that request as it may involve self-harm or dangerous activities.

If you or someone you know is in crisis, please reach out immediately:
• Emergency Services: 115 (Pakistan) / 911
• Visit your nearest hospital emergency department
• Talk to a trusted person or mental health professional

Your safety matters. Please seek help. 💙



In [9]:
# ── ADVISORY QUERY — should be redirected ──
response_advisory = ask_medibot("Can you diagnose me with diabetes?")


🧑 User: Can you diagnose me with diabetes?
ℹ️  [Safety Filter] Advisory redirect applied.

ℹ️  I understand you're looking for specific guidance.

As a general health information assistant, I cannot provide personal diagnoses 
or specific prescriptions — that requires a qualified doctor who can examine you properly.

I can share general information about conditions, symptoms, or healthy habits.
Please consult a healthcare professional for your specific situation. 🩺



## Step 8: Multi-Turn Conversation (Conversational Agent)
The chatbot remembers previous messages — just like a real conversation.

In [ ]:
def run_conversation(queries):
    """
    Runs a multi-turn conversation with MediBot.
    The chatbot remembers all previous messages for context.
    """
    history = []
    
    for query in queries:
        # Get response
        reply = ask_medibot(query, conversation_history=history)
        
        # Add to history for context in next turn
        history.append({"role": "user",      "content": query})
        history.append({"role": "assistant",  "content": reply})
    
    return history


# Test multi-turn conversation
conversation = [
    "What is high cholesterol?",
    "What foods should I avoid for it?",     # follow-up — uses previous context
    "How long does it take to improve?",     # another follow-up
]

history = run_conversation(conversation)

## Step 9: Interactive Chat (Run in Notebook)
Type your own questions and chat with MediBot directly!

In [ ]:
def interactive_chat():
    """
    Interactive chat loop — type questions and get answers.
    Type 'quit' or 'exit' to stop.
    """
    print("="*55)
    print("       🩺 MediBot — General Health Assistant")
    print("  Type your health question | Type 'quit' to exit")
    print("="*55)
    
    history = []
    
    while True:
        try:
            user_input = input("\nYou: ").strip()
        except (EOFError, KeyboardInterrupt):
            print("\n\nGoodbye! Stay healthy! 💙")
            break
        
        if not user_input:
            continue
        
        if user_input.lower() in ['quit', 'exit', 'bye']:
            print("\nGoodbye! Stay healthy! 💙")
            break
        
        # Get response
        reply = ask_medibot(user_input, conversation_history=history)
        
        # Update history
        history.append({"role": "user",     "content": user_input})
        history.append({"role": "assistant", "content": reply})


# ── START INTERACTIVE CHAT ──
interactive_chat()

---
## Summary & Key Findings

| Feature | Implementation |
|---|---|
| LLM Used | Groq API — LLaMA 3.3 70B (free) |
| Prompt Engineering | System prompt with doctor persona + strict rules |
| Safety Filter Level 1 | Danger patterns — blocks harmful queries |
| Safety Filter Level 2 | Advisory patterns — redirects personal diagnosis requests |
| Example Queries | ✅ Both PDF examples tested |
| Conversational Agent | ✅ Multi-turn with memory |
| Interactive Chat | ✅ Real-time input/output loop |

### Skills Demonstrated:
- ✅ **Prompt design and testing** — custom system prompt with persona and rules
- ✅ **Using APIs for LLMs** — Groq API with LLaMA model
- ✅ **Safety handling** — 2-layer filter (danger + advisory)
- ✅ **Building conversational agents** — multi-turn chat with history